# Requirements check

In [1]:
import sys
import datetime
print(sys.version)

3.12.9 (main, Mar 17 2025, 21:36:21) [Clang 20.1.0 ]


In [2]:
import darts
print(darts.__version__)

0.39.0


In [3]:
!conda env create -f environment.yml

zsh:1: command not found: conda


In [4]:
!conda install requirements.txt

zsh:1: command not found: conda


# Install dependencies

In [5]:
# cell A: which Python executable and environment
import sys, os
print("sys.executable:", sys.executable)
print("sys.version:", sys.version.replace('\n',' '))
print("PATH contains CUDA:", any("CUDA" in p or "cudnn" in p.lower() for p in os.environ.get("PATH","").split(os.pathsep)))


sys.executable: /Users/ivannovikov/Desktop/short_model/.venv/bin/python
sys.version: 3.12.9 (main, Mar 17 2025, 21:36:21) [Clang 20.1.0 ]
PATH contains CUDA: False


In [6]:
# cell B: torch diagnostics (safe if torch absent)
try:
    import torch
    print("torch.__version__:", torch.__version__)
    print("torch.version.cuda:", torch.version.cuda)
    print("torch.cuda.is_available():", torch.cuda.is_available())
    print("cuda device count:", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("device name:", torch.cuda.get_device_name(0))
except Exception as e:
    print("import torch failed / torch not found:", e)


torch.__version__: 2.9.1
torch.version.cuda: None
torch.cuda.is_available(): False
cuda device count: 0


In [7]:
import torch

torch.cuda.is_available()

False

In [8]:
!pip install -r requirements.txt

zsh:1: command not found: pip


## Imports

In [9]:
from utils.constants import (
    CATEGORY_MAP,
    BEST_PREDICTORS_FOR_INDEX
)
from utils.utils import (
    plot_known_feature_value,
    plot_graphics_for_each_ts,
    get_month_beginnings
)
import numpy as np
from utils.parametrized_values import (
    form_df_future,
    load_models,
    form_input_to_forecasting,
    train_model_and_eval_res,
    get_prediction_for_ts,
    df_combined,
    macro_data
)
from utils.cluster_forecast import ClusterForecaster
from itertools import combinations
from typing import Type, Literal, Any
from darts import TimeSeries
import pandas as pd
import os
from dateutil.relativedelta import relativedelta

import warnings
warnings.filterwarnings('ignore')

/Users/ivannovikov/Desktop/short_model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


Index(['Unnamed: 0', 'USDRUB', 'Кукуруза', 'Молоко сырое',
       'Мясо птицы бройлеров в живом весе ',
       'Мясо крупного рогатого скота в живом весе ', 'Подсолнечник',
       'Пшеница 1-го класса', 'Пшеница 3-го класса', 'Пшеница 5-го класса',
       'Сахар (средняя цена по России)', 'Свинина в живом весе',
       'Соевое масло', 'Соевый шрот', 'Соя', 'Яйцо товарное',
       'Подсолнечное масло наливом (мировые цены)',
       'Бутилированное подсолнечное масло (рафинированное)',
       'Подсолнечный шрот ', 'Колбасы сырокопченые', 'Баранина в живом весе',
       'Баранина в убойном весе ', 'Какао-бобы (USD)', 'Какао-бобы',
       'Подсолнечное масло (наливом) не бутилированное, нерафинированное ',
       'Кокосовое масло (USD) ', 'Кокосовое масло ', 'Пальмовое масло (USD)',
       'Пальмовое масло ', 'Табак (USD)', 'Табак', 'Пшеница 4-го класса',
       'Минтай б/г, Владивосток руб./кг', 'Минтай б/г, Китай C&F $/т',
       'Сельдь н/р, Владивосток руб./кг', 'Огурцы тепличные, руб.

---
## Global parameters setting

In [10]:
print(f"Available columns to predict: {df_combined.columns}, {macro_data.columns}")

Available columns to predict: Index(['Unnamed: 0', 'USDRUB', 'Кукуруза', 'Молоко сырое',
       'Мясо птицы бройлеров в живом весе ',
       'Мясо крупного рогатого скота в живом весе ', 'Подсолнечник',
       'Пшеница 1-го класса', 'Пшеница 3-го класса', 'Пшеница 5-го класса',
       ...
       'Апатитовый концетрат (FOB Morocco)_norm',
       'Аммиак (FOB Черное море)_norm',
       'Аммиачная селитра (FOB Черное море)_norm',
       'Хлорид калия (CFR Ю-В Азия)_norm',
       'Капролактам импортный контракт (Тайвань и Ю. Корея) CFR Азия_norm',
       'Метанол_norm', 'Бензол, CFR Япония_norm', 'Этилен, CFR Китай_norm',
       'Удобрения и агрохимикаты_norm_sum', 'Органические химикаты_norm_sum'],
      dtype='object', length=110), Index(['Инфляция - Рост индекса цен производителей (RUB, eop PPI),',
       'Инфляция - Рост индекса потребительских цен в США, в долларах США (USD, eop CPI),',
       'Инфляция - Рост индекса цен производителей в США, в долларах США (USD, eop PPI),',
      

## Описание параметров запуска

### Основные параметры прогнозирования
- `PREDICTION_HORIZON = 12` - горизонт прогнозирования (месяцев)
- `PREDICTOR_LAG = 12` - лаг предикторов

### Режимы работы и даты
- `IS_BACKTEST: bool = True` - режим бэктеста (True - тестирование на исторических данных, False - реальный прогноз)
- `BACKTEST_DATE` - дата начала бэктеста
- `SINGLE_DATAPOINT_TO_PRED = "2026-04-14"` - целевая дата для прогноза

### Параметры моделей
- `TARGET_COL_SUFFIX = "_sum"` - суффикс целевой переменной
- `MODEL_NAME_FOR_FORECASTER = "nbeats"` - тип модели для прогнозирования
- `USE_FUTURE_COV` - использование будущих ковариат (автоматически определяется по типу модели)

### Временные параметры
- `FUTURE_DATE` - будущая дата значений
- `train_end` - конец обучающего периода
- `past_idx` - даты исторических данных для обучения
- `future_idx` - даты будущего периода для прогноза

### Параметры внешних факторов
- `FUTURE_FORECASTER_TS_NAME = "Инфляция, г/г"` - название макроэкономического показателя
- `FUTURE_FORECASTER_TS_VALUES` - известные будущие значения макропоказателя

### Управление моделями
- `MODEL_SAVE_PATH` - шаблон пути для сохранения моделей
- `PERSIST_MODEL = True` - флаг сохранения моделей на диск
- `REFIT_MODEL = True` - флаг переобучения моделей

In [11]:
SINGLE_DATAPOINT_TO_PRED = "2025-01-01"

IS_BACKTEST: bool = df_combined.index[-1] >= pd.to_datetime(SINGLE_DATAPOINT_TO_PRED)
BACKTEST_PREDICTION_RANGE = 12 if IS_BACKTEST else 0

new_dates = pd.date_range(start=df_combined.index[-1] + pd.DateOffset(months=1),
                      end=SINGLE_DATAPOINT_TO_PRED,
                      freq='MS') if not IS_BACKTEST else \
            pd.date_range(start=pd.to_datetime(SINGLE_DATAPOINT_TO_PRED) - pd.DateOffset(months=BACKTEST_PREDICTION_RANGE) + pd.DateOffset(months=1),
                      end=SINGLE_DATAPOINT_TO_PRED,
                      freq='MS')

PREDICTION_HORIZON = len(new_dates) if not IS_BACKTEST else BACKTEST_PREDICTION_RANGE


FUTURE_VALUES = {
    # "Кукуруза": 0.0,
    # "Пшеница 1-го класса": 0.0,
    # "Пшеница 3-го класса": 0.0,
    # "Пшеница 4-го класса": 0.0,
    # "Пшеница 5-го класса": 0.0,
    # "Пшеница 12,5% FOB Ново, $ т": 0.0,
    # "Подсолнечник": 20.8,
    # "Соя": 0.0,
    # "Рапс, руб. т": 0.0,
    # # Oils & Meals
    # "Подсолнечное масло наливом (мировые цены)": 0.0,
    # "Бутилированное подсолнечное масло (рафинированное)": 0.0,
    # "Подсолнечное масло (наливом) не бутилированное, нерафинированное ": 0.0,
    # "Подсолнечное масло (наливом) не бутилированное, не": 0.0,
    # "Подсолнечный шрот ": 0.0,
    # "Соевое масло": 0.0,
    # "Соевый шрот": 0.0,
    # "Рапсовое масло EU, $ т": 0.0,
    # "Кокосовое масло ": 0.0,
    # "Пальмовое масло ": 0.0,
    # # Livestock
    # "Молоко сырое": 0.0,
    # "Мясо птицы бройлеров в живом весе ": 0.0,
    # "Мясо крупного рогатого скота в живом весе ": 0.0,
    # "Свинина в живом весе": 0.0,
    # "Баранина в живом весе": 0.0,
    # "Баранина в убойном весе ": 0.0,
    # "Яйцо товарное": 0.0,
    # # Processed meat
    # "Колбасы сырокопченые": 0.0,
    # "Колбасы вареные": 0.0,
    # # Fish
    # "Минтай б г, Владивосток руб. кг": 0.0,
    # "Минтай б г, Китай C&F $ т": 0.0,
    # "Сельдь н р, Владивосток руб. кг": 0.0,
    # # Vegetables
    # "Огурцы тепличные, руб. кг": 0.0,
    # "Томаты тепличные, руб. кг": 0.0,
    # # Other
    # "Сахар (средняя цена по России)": 0.0,
    # "Мука пшеничная, руб. т": 0.0,
    # # Stimuli crops
    # "Какао-бобы": 0.0,
    # "Табак": 0.0,
    # # Chemicals - dip
    # "Карбамид (FOB Южный)": 0.0,
    # "Моноаммонийфосфат, MAP (FOB Балтика)": 0.0,
    # "Апатитовый концетрат (FOB Morocco)": 0.0,
    # "Аммиак (FOB Черное море)": 0.0,
    # "Аммиачная селитра (FOB Черное море)": 0.0,
    # "Хлорид калия (CFR Ю-В Азия)": 0.0,
    # # Chemicals - bump
    # "Капролактам импортный контракт (Тайвань и Ю. Корея) CFR Азия": 0.0,
    # "Метанол": 0.0,
    # "Бензол, CFR Япония": 0.0,
    # "Этилен, CFR Китай": 0.0,
    # # macros
    # "Инфляция - Рост индекса цен производителей (RUB, eop PPI),": 0.0,
    # "Инфляция - Рост индекса потребительских цен в США, в долларах США (USD, eop CPI),": 0.0,
    # "Инфляция - Рост индекса цен производителей в США, в долларах США (USD, eop PPI),": 0.0,
    # "Ключевая ставка, годовых": 12.3,
    # "Инфляция, г/г": 8.9,
    # "USDRUB": 0.0
}
selected_clusters = [CATEGORY_MAP[i] for i in FUTURE_VALUES.keys()]

if len(selected_clusters) != len(set(selected_clusters)):
    raise ValueError("The elements must belong to different clusters.")

FUTURE_FORECASTER_TS_NAME = list(FUTURE_VALUES.keys())  # ["Подсолнечник", "Ключевая ставка, годовых", "Инфляция, г/г"]
FUTURE_FORECASTER_TS_VALUES_RAW = list(FUTURE_VALUES.values())

BACKTEST_DATE = df_combined.index[-1] - pd.DateOffset(months=PREDICTION_HORIZON)
TS_FREQUENCY = "MS"
train_end = BACKTEST_DATE if not IS_BACKTEST else pd.to_datetime(SINGLE_DATAPOINT_TO_PRED) - pd.DateOffset(months=BACKTEST_PREDICTION_RANGE)


if IS_BACKTEST:
    past_idx = pd.date_range(start=pd.to_datetime(SINGLE_DATAPOINT_TO_PRED) - pd.DateOffset(months=BACKTEST_PREDICTION_RANGE) + pd.DateOffset(months=1),
                               periods=PREDICTION_HORIZON,
                               freq=TS_FREQUENCY)
    future_idx = None
    
else:
    past_idx = pd.date_range(start=train_end + pd.DateOffset(months=1),
                           periods=PREDICTION_HORIZON,
                           freq=TS_FREQUENCY)
    future_idx = pd.date_range(start=past_idx[-1] + pd.DateOffset(months=1),
                           periods=PREDICTION_HORIZON,
                           freq=TS_FREQUENCY)

PERSIST_MODEL = True
REFIT_MODEL = True

PREDICTOR_LAG = 12
MODEL_SAVE_PATH = "{model_name}_dataset/{model_name}_for_{category_name}" + f"__{PREDICTION_HORIZON}"
TARGET_COL_SUFFIX = "_norm_sum"
MODEL_NAME_FOR_FORECASTER: Literal["tftmodel", "nbeats", "naiveseasonal", "chronos", "randomforest"] = "randomforest"
USE_FUTURE_COV = MODEL_NAME_FOR_FORECASTER in ("tftmodel", "randomforest", "chronos")

if IS_BACKTEST:
    df_updated = df_combined
    macro_updated = macro_data
    FUTURE_FORECASTER_TS_VALUES = []
    for (ts_name, ts_val) in zip(FUTURE_FORECASTER_TS_NAME, FUTURE_FORECASTER_TS_VALUES_RAW):
        df_target = df_updated if ts_name in df_updated.columns else macro_updated
        df_target.loc[df_target.index > train_end, ts_name] = np.linspace(df_target[df_target.index == train_end].iloc[0][ts_name], ts_val, PREDICTION_HORIZON)
        FUTURE_FORECASTER_TS_VALUES.append(df_target[df_target.index > train_end][ts_name].values)
else:
    df_updated, macro_updated, FUTURE_FORECASTER_TS_VALUES = form_df_future(new_dates,
                                                                           FUTURE_FORECASTER_TS_NAME,
                                                                           FUTURE_FORECASTER_TS_VALUES_RAW,
                                                                           PREDICTION_HORIZON
                                                                       )

In [12]:
train_end

Timestamp('2024-01-01 00:00:00')

In [13]:
for ts_name, ts_val in zip(FUTURE_FORECASTER_TS_NAME, FUTURE_FORECASTER_TS_VALUES):
    plot_known_feature_value(ts_val,
                             ts_name,
                             pd.date_range(start=new_dates[0],
                               periods=PREDICTION_HORIZON,
                               freq="MS"))

---
## Forecasting clusers

### predictions

In [14]:
models = train_model_and_eval_res(df_updated,
                         macro_updated,
                         past_idx = past_idx,
                         future_idx = future_idx,
                         predict_up_to=SINGLE_DATAPOINT_TO_PRED,
                         category_map = CATEGORY_MAP,
                         best_predictors_for_idx = BEST_PREDICTORS_FOR_INDEX,
                         target_col_suffix = TARGET_COL_SUFFIX,
                         future_forecaster_ts_name = FUTURE_FORECASTER_TS_NAME,
                         future_forecaster_ts_values = FUTURE_FORECASTER_TS_VALUES,
                         model_name = MODEL_NAME_FOR_FORECASTER,
                         use_future_macro = USE_FUTURE_COV,
                         prediction_horizon = PREDICTION_HORIZON,
                         dir_to_save_plots = f"darts_result_pairs_{MODEL_NAME_FOR_FORECASTER}",
                         dir_to_save_tables = f"darts_result_tables_pairs_{MODEL_NAME_FOR_FORECASTER}",
                         persist_model=PERSIST_MODEL,
                         refit=REFIT_MODEL,
                         prediction_lag=PREDICTOR_LAG,
                         is_backtest=IS_BACKTEST,
                         ts_freq=TS_FREQUENCY,
                         pred_index=new_dates
)

Saved forecaster to randomforest_dataset/randomforest_for_USDRUB__12.pkl and model to randomforest_dataset/randomforest_for_USDRUB__12_model.pth


Saved forecaster to randomforest_dataset/randomforest_for_Животноводство и мясо__12.pkl and model to randomforest_dataset/randomforest_for_Животноводство и мясо__12_model.pth


Saved forecaster to randomforest_dataset/randomforest_for_Зерно и семена__12.pkl and model to randomforest_dataset/randomforest_for_Зерно и семена__12_model.pth


ValueError: `actual_series` must be strictly positive to compute the MAPE.


Could not calculate mape for prediction due to `actual_series` must be strictly positive to compute the MAPE.


...

Saved forecaster to randomforest_dataset/randomforest_for_Переработанное мясо__12.pkl and model to randomforest_dataset/randomforest_for_Переработанное мясо__12_model.pth


Saved forecaster to randomforest_dataset/randomforest_for_Рыба__12.pkl and model to randomforest_dataset/randomforest_for_Рыба__12_model.pth


Saved forecaster to randomforest_dataset/randomforest_for_Сахар и мука__12.pkl and model to randomforest_dataset/randomforest_for_Сахар и мука__12_model.pth


Saved forecaster to randomforest_dataset/randomforest_for_Стимулирующие культуры__12.pkl and model to randomforest_dataset/randomforest_for_Стимулирующие культуры__12_model.pth


Saved forecaster to randomforest_dataset/randomforest_for_Удобрения и агрохимикаты__12.pkl and model to randomforest_dataset/randomforest_for_Удобрения и агрохимикаты__12_model.pth


---
## Inference model

In [15]:
print(MODEL_SAVE_PATH)
models = load_models(MODEL_NAME_FOR_FORECASTER, MODEL_SAVE_PATH)

{model_name}_dataset/{model_name}_for_{category_name}__12
[OK] Loaded forecaster ← randomforest_dataset/randomforest_for_Удобрения и агрохимикаты__12.pkl
[OK] Loaded model      ← randomforest_dataset/randomforest_for_Удобрения и агрохимикаты__12_model.pth
[OK] Loaded forecaster ← randomforest_dataset/randomforest_for_Инфляция - Рост индекса цен производителей (RUB, eop PPI),__12.pkl
[OK] Loaded model      ← randomforest_dataset/randomforest_for_Инфляция - Рост индекса цен производителей (RUB, eop PPI),__12_model.pth
[OK] Loaded forecaster ← randomforest_dataset/randomforest_for_Органические химикаты__12.pkl
[OK] Loaded model      ← randomforest_dataset/randomforest_for_Органические химикаты__12_model.pth
[OK] Loaded forecaster ← randomforest_dataset/randomforest_for_Переработанное мясо__12.pkl
[OK] Loaded model      ← randomforest_dataset/randomforest_for_Переработанное мясо__12_model.pth
[OK] Loaded forecaster ← randomforest_dataset/randomforest_for_Животноводство и мясо__12.pkl
[OK

In [16]:
from sklearn.metrics import mean_absolute_percentage_error as mape


answer_last = pd.DataFrame([])
df_updated.columns = [col_name.replace("/", " ") for col_name in df_updated.columns]
mape_values = {}
processed_columns = []
for col in list(CATEGORY_MAP.keys()):
    is_predicting_macro = col in macro_updated.columns
    df_for_col = macro_updated if is_predicting_macro else df_updated
    if col in ('Подсолнечное масло (наливом) не бутилированное, не'):
        continue
    category_to_pred = CATEGORY_MAP.get(col, "")
    if category_to_pred == "":
        continue

    cluster_preds, ts_prediction = get_prediction_for_ts(df_updated,
                                                          models[category_to_pred],
                                                          col,
                                                          MODEL_NAME_FOR_FORECASTER,
                                                          macro_updated,
                                                          past_idx,
                                                          future_idx,
                                                          CATEGORY_MAP,
                                                          BEST_PREDICTORS_FOR_INDEX,
                                                          TARGET_COL_SUFFIX,
                                                          FUTURE_FORECASTER_TS_NAME,
                                                          FUTURE_FORECASTER_TS_VALUES,
                                                          USE_FUTURE_COV,
                                                          PREDICTION_HORIZON,
                                                          IS_BACKTEST,
                                                          PREDICTOR_LAG,
                                                          new_dates
                                                        )
    processed_columns.append(col)
    answer_last = pd.concat([answer_last, ts_prediction], axis=1)
    if not is_predicting_macro:
        mape_val = mape(df_for_col.loc[ts_prediction.index][col], ts_prediction[f"{col}_preds"]) if IS_BACKTEST else 0
    else:
        mape_val = None
    mape_values[col] = mape_val
    plot_graphics_for_each_ts(df_for_col[f"{col}"],
                          ts_prediction[f"{col}_preds"],
                          None,
                          col,
                          mape_val,
                          -1,
                          BEST_PREDICTORS_FOR_INDEX[CATEGORY_MAP[col]],
                          "",
                          forecasting_horizon=PREDICTION_HORIZON)

[]
2024-01-01 00:00:00


[]
2024-01-01 00:00:00


[]
2024-01-01 00:00:00


[]
2024-01-01 00:00:00


[]
2024-01-01 00:00:00


...

[]


[]


[]


[]


[]
2024-01-01 00:00:00


In [17]:
exclude = ['Подсолнечное масло (наливом) не бутилированное, не', 'Инфляция - Рост индекса цен производителей (RUB, eop PPI),', 'Инфляция - Рост индекса потребительских цен в США, в долларах США (USD, eop CPI),', 'Инфляция - Рост индекса цен производителей в США, в долларах США (USD, eop PPI),', 'Ключевая ставка, годовых', 'Инфляция, г/г']
result = [x for x in list(CATEGORY_MAP.keys()) if x not in exclude]

answer_last = answer_last.filter(regex='_preds')
answer_last = answer_last.loc[:,~answer_last.columns.duplicated()].copy()
answer_last.columns = [i.replace('_preds', '') for i in answer_last.columns]

col_to_keep = df_updated[result][list(set(FUTURE_FORECASTER_TS_NAME) - set(macro_data))]
to_save = answer_last.reindex(df_updated[result].index).combine_first(df_updated[result])
to_save[list(set(FUTURE_FORECASTER_TS_NAME) - set(macro_data))] = col_to_keep
to_save = macro_data.reindex(to_save.index).combine_first(to_save)

to_save = to_save[list(mape_values.keys())]

In [18]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

file_name = f"prediction_{MODEL_NAME_FOR_FORECASTER}_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.xlsx"
to_save.to_excel(file_name)

wb = load_workbook(file_name)
ws = wb.active

first_idx = answer_last.index[0]
row_num = list(to_save.index).index(first_idx) + 2 

cols_to_fill = list(FUTURE_VALUES.keys())
col_indices = [idx + 1 for idx, cell in enumerate(ws[1]) if cell.value in cols_to_fill]

fill_yellow = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
for col in range(1, ws.max_column + 1):
    ws.cell(row=row_num, column=col).fill = fill_yellow

fill_red = PatternFill(start_color="FF0000", end_color="FF0000", fill_type="solid")
for col in col_indices:
    for row in range(row_num, ws.max_row + 1):
        ws.cell(row=row, column=col).fill = fill_red

wb.save(file_name)


In [19]:
import pandas as pd
import numpy as np
from openpyxl import Workbook

def save_pointwise_metrics(real_data: pd.DataFrame, predictions: pd.DataFrame, excel_path="metrics.xlsx"):
    if real_data.shape != predictions.shape:
        raise ValueError("real_data и predictions должны иметь одинаковую форму")

    # Поточечный MAPE
    mape_pointwise = (np.abs(real_data.values - predictions.values) / real_data.values) * 100
    mape_df = pd.DataFrame(mape_pointwise, index=predictions.index, columns=predictions.columns)

    # Поточечный SMAPE
    smape_pointwise = (2 * np.abs(real_data.values - predictions.values) / 
                       (np.abs(real_data.values) + np.abs(predictions.values))) * 100
    smape_df = pd.DataFrame(smape_pointwise, index=predictions.index, columns=predictions.columns)

    wb = Workbook()

    # Лист MAPE
    ws_mape = wb.active
    ws_mape.title = "MAPE"
    ws_mape.append(["Index"] + list(mape_df.columns))
    for idx, row in mape_df.iterrows():
        ws_mape.append([idx] + row.tolist())

    # Лист SMAPE
    ws_smape = wb.create_sheet(title="SMAPE")
    ws_smape.append(["Index"] + list(smape_df.columns))
    for idx, row in smape_df.iterrows():
        ws_smape.append([idx] + row.tolist())

    # Сохраняем
    wb.save(excel_path)

In [20]:
if IS_BACKTEST:
    real_matrix = pd.concat(
    [df_combined.loc[(df_combined.index <= past_idx[-1]) & (df_combined.index > past_idx[-1] - pd.DateOffset(months=BACKTEST_PREDICTION_RANGE))],
     macro_data.loc[df_combined[(df_combined.index <= past_idx[-1]) & (df_combined.index > past_idx[-1] - pd.DateOffset(months=BACKTEST_PREDICTION_RANGE))].index]],
    axis=1
    )[answer_last.columns]

    mape_filename = f"metrics_pointwise_{MODEL_NAME_FOR_FORECASTER}_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.xlsx"

    real_quarterly = real_matrix.resample('Q').mean()
    pred_quarterly = answer_last.resample('Q').mean()

    real_yearly = real_matrix.resample('Y').mean()
    pred_yearly = answer_last.resample('Y').mean()

    save_pointwise_metrics(real_data=real_matrix, predictions=answer_last, excel_path=f'M_{mape_filename}')
    save_pointwise_metrics(real_data=real_quarterly, predictions=pred_quarterly, excel_path=f'Q_{mape_filename}')
    save_pointwise_metrics(real_data=real_yearly, predictions=pred_yearly, excel_path=f'Y_{mape_filename}')